# Лабораторная работа №2  
## Сравнительный анализ свёрточных архитектур

В работе исследуются:

- MLP, LeNet и AlexNet;
- предобученная ResNet50;
- ResNet50 с блоками SE и ECA;
- inverted bottleneck с различными значениями `expand_ratio`.

Этот notebook содержит служебные функции и заготовки классов. Реализации архитектурных блоков, полный цикл обучения, экспериментальные результаты и выводы необходимо подготовить самостоятельно.

Подключим библиотеки, зафиксируем начальное состояние генераторов
случайных чисел и выберем устройство. Полная воспроизводимость между
разными версиями библиотек и разным оборудованием не гарантируется.

In [ ]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter
import random

import matplotlib.pyplot as plt
import numpy as np
import torch

from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import ResNet50_Weights, resnet50


SEED = 42
BATCH_SIZE = 8

DATASET_NAME = "CIFAR10"
CSU101_NUM_CLASSES = None

if DATASET_NAME == "CIFAR10":
    NUM_CLASSES = 10
elif DATASET_NAME == "CSU101":
    if CSU101_NUM_CLASSES is None:
        raise ValueError(
            "Укажите число классов выбранной версии CSU101"
        )
    NUM_CLASSES = CSU101_NUM_CLASSES
else:
    raise ValueError(
        f"Неизвестный датасет: {DATASET_NAME}"
    )

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


def set_reproducibility(seed: int) -> torch.Generator:
    """Настройка генераторов случайных чисел."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    generator = torch.Generator()
    generator.manual_seed(seed)

    return generator


data_generator = set_reproducibility(SEED)

print("Версия PyTorch:", torch.__version__)
print("Датасет:", DATASET_NAME)
print("Число классов:", NUM_CLASSES)
print("Устройство:", device)

## 1. Проверка данных и окружения

Для быстрой проверки используется `FakeData`. Эти изображения нужны
только для контроля размерностей и запуска модели. Полученные на них
значения loss, accuracy и FPS не включаются в итоговый отчёт.

Размер `224 × 224` выбран для стандартной ResNet50. При сравнении
моделей необходимо явно указать размер входа каждой архитектуры.
FPS моделей с разными размерами входа нельзя сопоставлять без
соответствующей оговорки.

In [ ]:
SMOKE_INPUT_SIZE = (224, 224)

smoke_transform = transforms.Compose(
    [
        transforms.Resize(SMOKE_INPUT_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        ),
    ]
)

smoke_dataset = datasets.FakeData(
    size=32,
    image_size=(3, *SMOKE_INPUT_SIZE),
    num_classes=NUM_CLASSES,
    transform=smoke_transform,
    random_offset=SEED,
)

smoke_loader = DataLoader(
    smoke_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=data_generator,
)

images, targets = next(iter(smoke_loader))

print("Размер изображений:", tuple(images.shape))
print("Размер меток:", tuple(targets.shape))

## 2. Подготовка основного датасета

 Основной эксперимент выполняется на CIFAR-10 либо CSU101.

 Для всех моделей фиксируются:

- одно и то же разбиение train/validation/test;
- одинаковый способ выбора лучшей эпохи;
- одинаковые правила использования test-набора;
- документированные преобразования изображений;
- одинаковый seed.

Validation-набор используется для выбора гиперпараметров.
Test-набор применяется после завершения настройки моделей.

Для предобученной ResNet50 обычно используется нормализация ImageNet.
Для MLP, LeNet и AlexNet размер входа может отличаться. Это различие
необходимо указать в итоговой таблице.

Пример загрузки CIFAR-10:

```python
train_transform = transforms.Compose([...])
eval_transform = transforms.Compose([...])

full_train_dataset = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=train_transform,
)

test_dataset = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=eval_transform,
)

train_dataset, validation_dataset = ...
```
#
Случайные аугментации применяются только к обучающей части.
Разбиение необходимо выполнить с использованием `data_generator`.

## 3. Подготовка ResNet50

Функция создаёт ResNet50 и заменяет классификационную голову.
В основном эксперименте используются предобученные веса.

При отсутствии доступа к интернету веса нужно заранее загрузить
в кэш torchvision.

Рекомендуемый начальный learning rate для fine-tuning - `1e-4`.
Число эпох, scheduler, weight decay и стратегия разморозки выбираются
и обосновываются в отчёте.

In [ ]:
def build_resnet50(
    num_classes: int,
    use_pretrained: bool,
) -> nn.Module:
    """Создание ResNet50 с новой классификационной головой."""
    weights = (
        ResNet50_Weights.DEFAULT
        if use_pretrained
        else None
    )

    model = resnet50(weights=weights)

    model.fc = nn.Linear(
        model.fc.in_features,
        num_classes,
    )

    return model


# Для проверки используется модель без скачивания весов.
smoke_model = build_resnet50(
    num_classes=NUM_CLASSES,
    use_pretrained=False,
).to(device)

# В основном эксперименте:
#
# model = build_resnet50(
#     num_classes=NUM_CLASSES,
#     use_pretrained=True,
# ).to(device)

print(smoke_model.fc)

Следующая функция проверяет форму выхода модели. Она полезна после
замены классификационной головы и после добавления SE или ECA.

In [ ]:
def check_model_output(
    model: nn.Module,
    example_batch: torch.Tensor,
    num_classes: int,
    target_device: torch.device,
) -> None:
    """Проверка формы выхода классификатора."""
    model.eval()
    batch = example_batch.to(target_device)

    with torch.inference_mode():
        output = model(batch)

    expected_shape = (
        batch.shape[0],
        num_classes,
    )

    if tuple(output.shape) != expected_shape:
        raise ValueError(
            f"Ожидалась форма {expected_shape}, "
            f"получена {tuple(output.shape)}"
        )


check_model_output(
    smoke_model,
    images,
    NUM_CLASSES,
    device,
)

## 4. Начальный этап fine-tuning

На первом этапе можно обучать только новую классификационную голову.
Затем выбранную часть backbone размораживают и продолжают обучение
с меньшим learning rate.

Для моделей с SE/ECA необходимо отдельно проверить, что новые
attention-блоки не оказались заморожены вместе с backbone.

In [ ]:
def freeze_backbone(
    model: nn.Module,
) -> None:
    """Заморозка всех параметров, кроме классификационной головы."""
    for parameter in model.parameters():
        parameter.requires_grad = False

    for parameter in model.fc.parameters():
        parameter.requires_grad = True


freeze_backbone(smoke_model)

trainable_parameters = sum(
    parameter.numel()
    for parameter in smoke_model.parameters()
    if parameter.requires_grad
)

print(
    "Обучаемых параметров в проверочной модели:",
    trainable_parameters,
)

Один шаг оптимизации позволяет проверить совместимость модели,
данных, функции потерь и оптимизатора. Он не является полноценным
обучением.

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    (
        parameter
        for parameter in smoke_model.parameters()
        if parameter.requires_grad
    ),
    lr=1e-4,
)

smoke_model.train()
optimizer.zero_grad(set_to_none=True)

logits = smoke_model(images.to(device))
loss = criterion(logits, targets.to(device))

loss.backward()
optimizer.step()

print(
    "Loss после проверочного шага:",
    float(loss.detach().cpu()),
)

Для основной работы порядок заморозки и разморозки параметров нужно
задавать явно. Функция ниже должна учитывать backbone, attention-блоки
и классификационную голову.

In [ ]:
def configure_trainable_parameters(
    model: nn.Module,
    train_attention: bool,
    train_classifier: bool,
    train_backbone: bool,
) -> None:
    """Настройка обучаемых частей модели."""
    raise NotImplementedError

## 5. Контрольные архитектуры

MLP, LeNet и AlexNet нужны для анализа влияния глубины модели.
Архитектуры обучаются на одном разбиении данных и сравниваются по
одинаковому протоколу.

Для каждой модели необходимо указать:

- размер входного изображения;
- число эпох;
- optimizer и learning rate;
- dropout и weight decay;
- правило выбора лучшей эпохи.

In [ ]:
class MLPBaseline(nn.Module):
    """Полносвязная контрольная модель."""

    def __init__(
        self,
        num_classes: int,
        input_shape: tuple[int, int, int],
    ) -> None:
        super().__init__()
        raise NotImplementedError

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        raise NotImplementedError


class LeNetBaseline(nn.Module):
    """Свёрточная сеть на основе LeNet."""

    def __init__(
        self,
        num_classes: int,
        input_size: tuple[int, int],
    ) -> None:
        super().__init__()
        raise NotImplementedError

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        raise NotImplementedError


class AlexNetBaseline(nn.Module):
    """Свёрточная сеть на основе AlexNet."""

    def __init__(
        self,
        num_classes: int,
        input_size: tuple[int, int],
    ) -> None:
        super().__init__()
        raise NotImplementedError

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        raise NotImplementedError

## 6. Исследование глубины и регуляризации

Для анализа регуляризации за один эксперимент изменяется только один
фактор: dropout, weight decay либо состав аугментаций.

Для каждой эпохи сохраняются:

- train loss;
- validation loss;
- train accuracy;
- validation accuracy.

Test accuracy не используется для выбора конфигурации.

In [ ]:
regularization_experiments = [
    {
        "name": "базовый вариант",
        "dropout": None,
        "weight_decay": None,
        "augmentation": None,
    },
    {
        "name": "вариант регуляризации 1",
        "dropout": None,
        "weight_decay": None,
        "augmentation": None,
    },
    {
        "name": "вариант регуляризации 2",
        "dropout": None,
        "weight_decay": None,
        "augmentation": None,
    },
]

## 7. SE и ECA

SE формирует канальные веса по схеме: `global average pooling → два FC-слоя → sigmoid`.

ECA использует: `global average pooling → Conv1d → sigmoid`.

Число каналов attention-блока должно совпадать с числом каналов
выхода соответствующего bottleneck.

Для SE необходимо проверить положительность `channels` и `reduction`.
Для ECA размер ядра должен быть положительным и нечётным.

In [ ]:
class SEBlock(nn.Module):
    """Канальное внимание Squeeze-and-Excitation."""

    def __init__(
        self,
        channels: int,
        reduction: int = 16,
    ) -> None:
        super().__init__()

        if channels <= 0:
            raise ValueError(
                "Число каналов должно быть положительным"
            )

        if reduction <= 0:
            raise ValueError(
                "Коэффициент reduction должен быть положительным"
            )

        raise NotImplementedError

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        raise NotImplementedError


class ECABlock(nn.Module):
    """Канальное внимание Efficient Channel Attention."""

    def __init__(
        self,
        channels: int,
        kernel_size: int = 3,
    ) -> None:
        super().__init__()

        if channels <= 0:
            raise ValueError(
                "Число каналов должно быть положительным"
            )

        if kernel_size <= 0 or kernel_size % 2 == 0:
            raise ValueError(
                "Размер ядра должен быть положительным и нечётным"
            )

        raise NotImplementedError

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        raise NotImplementedError

По условию работы SE или ECA размещаются после каждого bottleneck.
Во всех сравниваемых вариантах место вставки должно быть одинаковым.
После модификации модели необходимо повторно проверить форму выхода.

In [ ]:
def insert_attention_after_bottlenecks(
    model: nn.Module,
    attention_type: str,
) -> nn.Module:
    """Добавление SE или ECA после bottleneck-блоков ResNet."""
    raise NotImplementedError(
        "Допишите обход модели и вставку attention-блоков"
    )

## 8. Inverted bottleneck

Блок состоит из:

1. расширения числа каналов;
2. depthwise-свёртки;
3. линейной проекции.

Остаточное соединение применяется при `stride = 1` и совпадении
числа входных и выходных каналов.

Необходимо проверить:
- expand_ratio = 3;
- expand_ratio = 6;
- expand_ratio = 12.

In [ ]:
class InvertedBottleneck(nn.Module):
    """Блок с расширением, depthwise-свёрткой и проекцией."""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int,
        expand_ratio: int,
    ) -> None:
        super().__init__()

        if in_channels <= 0 or out_channels <= 0:
            raise ValueError(
                "Число каналов должно быть положительным"
            )

        if stride not in {1, 2}:
            raise ValueError(
                "Для этой работы stride должен быть равен 1 или 2"
            )

        if expand_ratio <= 0:
            raise ValueError(
                "expand_ratio должен быть положительным"
            )

        raise NotImplementedError

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        raise NotImplementedError

При сравнении коэффициентов расширения остальные части архитектуры
должны оставаться одинаковыми.

In [ ]:
def build_inverted_bottleneck_model(
    num_classes: int,
    expand_ratio: int,
) -> nn.Module:
    """Сборка классификатора из inverted bottleneck."""
    raise NotImplementedError


expand_ratios = [3, 6, 12]

for ratio in expand_ratios:
    print(
        f"Запланирован эксперимент: expand_ratio={ratio}"
    )

## 9. Обучение и валидация

Для всех архитектур используются единые функции обучения и оценки.
Функции возвращают loss и accuracy, чтобы результаты можно было
сохранять в общей истории.

In [ ]:
def train_one_epoch(
    model: nn.Module,
    data_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    target_device: torch.device,
) -> dict[str, float]:
    """
    Обучение модели в течение одной эпохи.

    Возвращает:
    {
        "loss": среднее значение функции потерь,
        "accuracy": доля правильных ответов,
    }
    """
    raise NotImplementedError


def evaluate_model(
    model: nn.Module,
    data_loader: DataLoader,
    criterion: nn.Module,
    target_device: torch.device,
) -> dict[str, float]:
    """
    Оценка модели без изменения параметров.

    Возвращает:
    {
        "loss": среднее значение функции потерь,
        "accuracy": доля правильных ответов,
    }
    """
    raise NotImplementedError

История обучения хранится отдельно для каждой модели. Она используется
для графиков train/validation loss и accuracy.

In [ ]:
def create_empty_history() -> dict[str, list[float]]:
    """Создание пустой истории обучения."""
    return {
        "train_loss": [],
        "validation_loss": [],
        "train_accuracy": [],
        "validation_accuracy": [],
    }


history = create_empty_history()

Лучшая эпоха выбирается по validation loss или validation accuracy.
Test-набор не используется для выбора checkpoint.

In [ ]:
def save_checkpoint(
    model: nn.Module,
    path: Path,
) -> None:
    """Сохранение параметров выбранной модели."""
    raise NotImplementedError

## 10. Подсчёт параметров

Для отчёта полезно различать общее число параметров и число
обучаемых параметров.

In [ ]:
def count_parameters(
    model: nn.Module,
) -> dict[str, int]:
    """Подсчёт всех и обучаемых параметров."""
    return {
        "total": sum(
            parameter.numel()
            for parameter in model.parameters()
        ),
        "trainable": sum(
            parameter.numel()
            for parameter in model.parameters()
            if parameter.requires_grad
        ),
    }


smoke_parameter_count = count_parameters(smoke_model)

print(
    "Параметры проверочной ResNet50:",
    smoke_parameter_count,
)

## 11. Измерение FPS

In [ ]:
def measure_model_fps(
    model: nn.Module,
    example_batch: torch.Tensor,
    target_device: torch.device,
    repeats: int = 10,
    warmup_runs: int = 2,
) -> float:
    """Измерение скорости инференса."""
    if repeats <= 0:
        raise ValueError(
            "Число повторов должно быть положительным"
        )

    if warmup_runs < 0:
        raise ValueError(
            "Число прогревочных запусков не может быть отрицательным"
        )

    model.eval()
    batch = example_batch.to(target_device)

    with torch.inference_mode():
        for _ in range(warmup_runs):
            model(batch)

        if target_device.type == "cuda":
            torch.cuda.synchronize()

        start_time = perf_counter()

        for _ in range(repeats):
            model(batch)

        if target_device.type == "cuda":
            torch.cuda.synchronize()

    elapsed = perf_counter() - start_time

    if elapsed <= 0:
        raise RuntimeError(
            "Не удалось корректно измерить время"
        )

    number_of_images = repeats * batch.shape[0]

    return number_of_images / elapsed


runtime_check_fps = measure_model_fps(
    smoke_model,
    images,
    device,
    repeats=2,
)

print(
    "Проверочная скорость:",
    round(runtime_check_fps, 2),
    "изображений/с",
)

## 12. Карты активаций

Карты активаций нужны для качественного сравнения базовой ResNet50 и вариантов с SE/ECA. Для всех моделей желательно использовать один и тот же слой и одни и те же изображения.

In [ ]:
def collect_activation_map(
    model: nn.Module,
    layer: nn.Module,
    image: torch.Tensor,
    target_device: torch.device,
) -> torch.Tensor:
    """Получение карты активаций выбранного слоя."""
    raise NotImplementedError


def plot_activation_map(
    image: torch.Tensor,
    activation: torch.Tensor,
    title: str,
) -> None:
    """Отображение изображения и карты активаций."""
    raise NotImplementedError

## 13. Итоговые результаты

Результаты всех экспериментов удобно собирать в одной таблице. Пустые значения следует заменить фактическими показателями, полученными на тестовой выборке.

In [ ]:
results = [
    {
        "model": "MLP",
        "input_size": None,
        "regularization": None,
        "validation_accuracy": None,
        "test_accuracy": None,
        "total_parameters": None,
        "trainable_parameters": None,
        "fps": None,
    },
    {
        "model": "LeNet",
        "input_size": None,
        "regularization": None,
        "validation_accuracy": None,
        "test_accuracy": None,
        "total_parameters": None,
        "trainable_parameters": None,
        "fps": None,
    },
    {
        "model": "AlexNet",
        "input_size": None,
        "regularization": None,
        "validation_accuracy": None,
        "test_accuracy": None,
        "total_parameters": None,
        "trainable_parameters": None,
        "fps": None,
    },
    {
        "model": "ResNet50",
        "input_size": None,
        "regularization": None,
        "validation_accuracy": None,
        "test_accuracy": None,
        "total_parameters": None,
        "trainable_parameters": None,
        "fps": None,
    },
    {
        "model": "ResNet50 + SE",
        "input_size": None,
        "regularization": None,
        "validation_accuracy": None,
        "test_accuracy": None,
        "total_parameters": None,
        "trainable_parameters": None,
        "fps": None,
    },
    {
        "model": "ResNet50 + ECA",
        "input_size": None,
        "regularization": None,
        "validation_accuracy": None,
        "test_accuracy": None,
        "total_parameters": None,
        "trainable_parameters": None,
        "fps": None,
    },
    {
        "model": "Inverted bottleneck, ratio=3",
        "input_size": None,
        "regularization": None,
        "validation_accuracy": None,
        "test_accuracy": None,
        "total_parameters": None,
        "trainable_parameters": None,
        "fps": None,
    },
    {
        "model": "Inverted bottleneck, ratio=6",
        "input_size": None,
        "regularization": None,
        "validation_accuracy": None,
        "test_accuracy": None,
        "total_parameters": None,
        "trainable_parameters": None,
        "fps": None,
    },
    {
        "model": "Inverted bottleneck, ratio=12",
        "input_size": None,
        "regularization": None,
        "validation_accuracy": None,
        "test_accuracy": None,
        "total_parameters": None,
        "trainable_parameters": None,
        "fps": None,
    },
]

Следующая функция выводит заполненную часть таблицы в текстовом виде.

In [ ]:
def print_results(
    experiment_results: list[dict],
) -> None:
    """Текстовый вывод результатов экспериментов."""
    header = (
        f"{'Модель':<35}"
        f"{'Val acc':>10}"
        f"{'Test acc':>12}"
        f"{'Параметры':>14}"
        f"{'FPS':>10}"
    )

    print(header)
    print("-" * len(header))

    for row in experiment_results:
        val_accuracy = row["validation_accuracy"]
        test_accuracy = row["test_accuracy"]
        parameters = row["total_parameters"]
        fps = row["fps"]

        val_text = (
            f"{val_accuracy:.4f}"
            if val_accuracy is not None
            else "-"
        )
        test_text = (
            f"{test_accuracy:.4f}"
            if test_accuracy is not None
            else "-"
        )
        parameters_text = (
            str(parameters)
            if parameters is not None
            else "-"
        )
        fps_text = (
            f"{fps:.2f}"
            if fps is not None
            else "-"
        )

        print(
            f"{row['model']:<35}"
            f"{val_text:>10}"
            f"{test_text:>12}"
            f"{parameters_text:>14}"
            f"{fps_text:>10}"
        )


print_results(results)

## 14. График accuracy - число параметров

После заполнения таблицы можно построить зависимость accuracy от количества параметров. На графике должны использоваться только результаты, полученные в одинаковых условиях.

In [ ]:
def plot_accuracy_vs_parameters(
    experiment_results: list[dict],
) -> None:
    """График зависимости test accuracy от числа параметров."""
    valid_rows = [
        row
        for row in experiment_results
        if (
            row["test_accuracy"] is not None
            and row["total_parameters"] is not None
        )
    ]

    if not valid_rows:
        print(
            "Сначала заполните результаты экспериментов"
        )
        return

    parameter_values = [
        row["total_parameters"]
        for row in valid_rows
    ]
    accuracy_values = [
        row["test_accuracy"]
        for row in valid_rows
    ]

    plt.figure(figsize=(9, 6))
    plt.plot(
        parameter_values,
        accuracy_values,
        marker="o",
    )

    for row in valid_rows:
        plt.annotate(
            row["model"],
            (
                row["total_parameters"],
                row["test_accuracy"],
            ),
        )

    plt.xlabel("Количество параметров")
    plt.ylabel("Test accuracy")
    plt.title(
        "Зависимость качества от сложности модели"
    )
    plt.grid(True)
    plt.show()

#Правила использования внешних ресурсов
- Допускается использование PyTorch, torchvision, torchsummary, thop
- Запрещено прямое копирование готовых реализаций SE/ECA блоков из репозиториев
- Фиксируйте random seed для воспроизводимости